# SportIQ — Models, end to end

Rebuilds all three models from the exported parquet data. **Self-contained**: no imports from
the SportIQ codebase, no database, no API keys.

| Sport | Architecture | Target |
|---|---|---|
| Basketball (NBA) | XGBoost binary + isotonic | P(home win) |
| Football | 2 Poisson regressors → XGBoost multiclass | 1X2, then Over/Under by Poisson CDF |
| Tennis (ATP) | XGBoost binary + isotonic | P(home win) |

### The one rule that matters

Every rolling statistic is filtered to `GAME_DATE < fixture_date`. That **strict** inequality is
the leakage guard. Change it to `<=` anywhere and the model silently learns from the match it
is predicting, and every metric below becomes meaningless.

### Requirements

    pip install pandas numpy scikit-learn xgboost pyarrow matplotlib

In [2]:
%pip install xgboost

   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   - -------------------------------------- 2.6/69.5 MB 34.2 MB/s eta 0:00:02
   ----- ---------------------------------- 9.7/69.5 MB 34.4 MB/s eta 0:00:02
   --------- ------------------------------ 17.3/69.5 MB 34.4 MB/s eta 0:00:02
   -------------- ------------------------- 26.0/69.5 MB 36.3 MB/s eta 0:00:02
   -------------------- ------------------- 35.1/69.5 MB 38.1 MB/s eta 0:00:01
   ------------------------ --------------- 42.7/69.5 MB 37.5 MB/s eta 0:00:01
   ----------------------------- ---------- 51.4/69.5 MB 38.2 MB/s eta 0:00:01
   ---------------------------------- ----- 60.0/69.5 MB 38.7 MB/s eta 0:00:01
   ---------------------------------------  69.5/69.5 MB 39.4 MB/s eta 0:00:01
   ---------------------------------------- 69.5/69.5 MB 36.6 MB/s  0:00:02
Note: you may need to restart the kernel to use updated packages.


In [3]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import accuracy_score, brier_score_loss
import xgboost as xgb

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

DATA = Path(".")           # folder containing the exported parquet files
print("files found:", len(list(DATA.glob("*.parquet"))))

files found: 31


---
## 1. Shared helpers

`rolling_before` is the leakage guard in one place. Everything else builds on it.

In [ ]:
def rolling_before(team_games: pd.DataFrame, as_of, column: str, n: int = 5):
    """Mean of `column` over a team's last `n` matches STRICTLY BEFORE as_of.

    Returns None when there is no prior history — never a fabricated zero, because "no games
    played yet" and "averaged zero" are different things and the model should be able to tell
    them apart."""
    prior = team_games[team_games["GAME_DATE"] < as_of]
    if prior.empty:
        return None
    recent = prior.sort_values("GAME_DATE", ascending=False).head(n)
    values = recent[column].dropna()
    return float(values.mean()) if len(values) else None


def temporal_split(df, train_seasons, val_season, test_season):
    """Split by SEASON, never randomly. A random split would put future matches in train."""
    return (
        df[df.SEASON.isin(train_seasons)].copy(),
        df[df.SEASON == val_season].copy(),
        df[df.SEASON == test_season].copy(),
    )


def report(name, y_true, proba, baseline_rate=None):
    pred = (proba >= 0.5).astype(int)
    acc = accuracy_score(y_true, pred)
    brier = brier_score_loss(y_true, proba)
    print(f"{name}: accuracy={acc:.4f}  brier={brier:.4f}  n={len(y_true)}")
    if baseline_rate is not None:
        print(f"   baseline (always home) = {baseline_rate:.4f}   edge = {acc - baseline_rate:+.4f}")
    return acc, brier

---
## 2. Basketball (NBA) — the strongest model

68.57% test accuracy against a 55.43% always-home baseline in the production run. A single
XGBoost binary classifier; no draw exists, so the problem is genuinely two-class.

In [ ]:
nba = pd.read_parquet(DATA / "nba_game_log.parquet")
nba["GAME_DATE"] = pd.to_datetime(nba["GAME_DATE"])
print(nba.shape)
nba.head(3)

In [ ]:
def nba_features(games, home_id, away_id, as_of, season):
    """A reduced NBA vector — the rolling-form half of the production 16.

    Key-player availability and the moneyline are omitted here: the first needs the box-score
    join (see section 5) and the second is only sparsely populated. Everything present is
    computed exactly as production does it."""
    h = games[games.TEAM_ID == home_id]
    a = games[games.TEAM_ID == away_id]

    def rest(team_games):
        prior = team_games[team_games.GAME_DATE < as_of]
        return None if prior.empty else float((as_of - prior.GAME_DATE.max()).days)

    h2h = h[(h.OPPONENT_ID == away_id) & (h.GAME_DATE < as_of)]
    return {
        "rest_days_home": rest(h),
        "rest_days_away": rest(a),
        "back_to_back_home": 1.0 if (rest(h) or 99) <= 1 else 0.0,
        "back_to_back_away": 1.0 if (rest(a) or 99) <= 1 else 0.0,
        "last10_win_rate_home": rolling_before(h, as_of, "WON", 10),
        "last10_win_rate_away": rolling_before(a, as_of, "WON", 10),
        "last10_point_diff_home": rolling_before(h, as_of, "POINT_DIFF", 10),
        "last10_point_diff_away": rolling_before(a, as_of, "POINT_DIFF", 10),
        "home_court_indicator": 1.0,
        "h2h_win_rate_home": float(h2h.WON.mean()) if len(h2h) else None,
    }

In [ ]:
# Derive the two columns the features need, then assemble one row per game.
nba["WON"] = (nba["WL"] == "W").astype(int) if "WL" in nba.columns else nba["WON"]
if "POINT_DIFF" not in nba.columns:
    nba["POINT_DIFF"] = nba["PTS"] - nba["OPP_PTS"] if "OPP_PTS" in nba.columns else np.nan

home_rows = nba[nba.HOME_AWAY == "home"] if "HOME_AWAY" in nba.columns else nba
print("columns available:", list(nba.columns)[:14])

> **Note on schema.** The exported NBA log is `nba_api`'s own shape. Inspect the columns above
> and map them to `WON` / `POINT_DIFF` / `HOME_AWAY` before running the assembly loop — the
> production collector does this in `ml/training/collect_nba_data.py`.

---
## 3. Football — the two-layer model

The most involved of the three, because it answers two different questions: **who wins**
(3-way) and **how many goals** (which drives Over/Under). One classifier cannot do both.

    25 features → 2 Poisson regressors → xg_home, xg_away
                                       ├→ Layer 2 XGBoost → 1X2
                                       └→ Poisson CDF     → Over/Under

In [ ]:
LEAGUES = ["epl", "brasileirao", "mls", "csl", "scottish_prem"]

def load_football():
    frames = []
    for lg in LEAGUES:
        f = DATA / f"football_game_log_{lg}.parquet"
        if f.exists():
            frames.append(pd.read_parquet(f).assign(LEAGUE=lg))
    games = pd.concat(frames, ignore_index=True)
    games["GAME_DATE"] = pd.to_datetime(games["GAME_DATE"])

    def merge_kind(kind, value_col, out_for, out_against):
        parts = [pd.read_parquet(DATA / f"football_{kind}_{lg}.parquet")
                 for lg in LEAGUES if (DATA / f"football_{kind}_{lg}.parquet").exists()]
        if not parts:
            return games
        extra = pd.concat(parts, ignore_index=True)
        own = extra.rename(columns={value_col: out_for})[["FIXTURE_ID", "TEAM_ID", out_for]]
        opp = extra.rename(columns={"TEAM_ID": "OPPONENT_ID", value_col: out_against})[
            ["FIXTURE_ID", "OPPONENT_ID", out_against]]
        return games.merge(own, on=["FIXTURE_ID", "TEAM_ID"], how="left") \
                    .merge(opp, on=["FIXTURE_ID", "OPPONENT_ID"], how="left")

    return merge_kind, games

merge_kind, games = load_football()
games = merge_kind("xg", "XG_FOR", "XG_FOR", "XG_AGAINST")
merge_kind, games2 = (lambda k, v, a, b: games, games)  # xg merged; corners optional below
print(games.shape, "| xG present on", games.XG_FOR.notna().sum(), "rows")
games.head(3)

### Elo — the one genuinely stateful feature

Every other feature is an independently re-derivable rolling window. Elo needs a single
chronological pass over the whole log, which is why it is computed once up front rather than
per fixture.

In [ ]:
INITIAL_ELO, K_FACTOR = 1500.0, 32.0

def compute_elo_history(games):
    """{(FIXTURE_ID, TEAM_ID): elo BEFORE that match}. One pass, in date order."""
    ratings, history = {}, {}
    fixtures = (games.sort_values("GAME_DATE")
                     .groupby(["FIXTURE_ID", "GAME_DATE"], sort=True))
    for (fid, _date), group in fixtures:
        home = group[group.HOME_AWAY == "home"]
        away = group[group.HOME_AWAY == "away"]
        if home.empty or away.empty:
            continue
        h, a = home.iloc[0], away.iloc[0]
        rh = ratings.get(h.TEAM_ID, INITIAL_ELO)
        ra = ratings.get(a.TEAM_ID, INITIAL_ELO)
        history[(fid, h.TEAM_ID)], history[(fid, a.TEAM_ID)] = rh, ra   # BEFORE the result
        expected = 1.0 / (1.0 + 10 ** ((ra - rh) / 400.0))
        actual = 1.0 if h.GF > h.GA else (0.5 if h.GF == h.GA else 0.0)
        ratings[h.TEAM_ID] = rh + K_FACTOR * (actual - expected)
        ratings[a.TEAM_ID] = ra + K_FACTOR * ((1 - actual) - (1 - expected))
    return history

elo_history = compute_elo_history(games)
print("elo entries:", len(elo_history))

In [ ]:
POINTS = {"W": 3, "D": 1, "L": 0}

def football_features(games, home_id, away_id, as_of, fixture_id):
    h = games[games.TEAM_ID == home_id]
    a = games[games.TEAM_ID == away_id]

    def rest(tg):
        prior = tg[tg.GAME_DATE < as_of]
        return None if prior.empty else float((as_of - prior.GAME_DATE.max()).days)

    def form_pts(tg):
        prior = tg[tg.GAME_DATE < as_of].sort_values("GAME_DATE", ascending=False).head(5)
        return float(prior.WDL.map(POINTS).mean()) if len(prior) else None

    def streak(tg):
        prior = tg[tg.GAME_DATE < as_of].sort_values("GAME_DATE", ascending=False)
        run = 0
        for wdl in prior.WDL:
            if wdl != "W":
                break
            run += 1
        return float(run)

    h2h = h[(h.OPPONENT_ID == away_id) & (h.GAME_DATE < as_of)].head(10)
    eh, ea = elo_history.get((fixture_id, home_id)), elo_history.get((fixture_id, away_id))

    return {
        "attack_str_home": rolling_before(h, as_of, "GF"),
        "attack_str_away": rolling_before(a, as_of, "GF"),
        "defence_str_home": rolling_before(h, as_of, "GA"),
        "defence_str_away": rolling_before(a, as_of, "GA"),
        "form_pts_home": form_pts(h),
        "form_pts_away": form_pts(a),
        "rest_days_home": rest(h),
        "rest_days_away": rest(a),
        "h2h_win_rate_home": float((h2h.WDL == "W").mean()) if len(h2h) else None,
        "h2h_avg_goals_scored_home": float(h2h.GF.mean()) if len(h2h) else None,
        "h2h_avg_goals_allowed_home": float(h2h.GA.mean()) if len(h2h) else None,
        "elo_diff": (eh - ea) if (eh is not None and ea is not None) else None,
        "win_streak_home": streak(h),
        "win_streak_away": streak(a),
        "xg_for_home": rolling_before(h, as_of, "XG_FOR"),
        "xg_against_home": rolling_before(h, as_of, "XG_AGAINST"),
        "xg_for_away": rolling_before(a, as_of, "XG_FOR"),
        "xg_against_away": rolling_before(a, as_of, "XG_AGAINST"),
    }

FOOTBALL_FEATURES = list(football_features(games, games.TEAM_ID.iloc[0],
                                           games.OPPONENT_ID.iloc[0],
                                           games.GAME_DATE.max(), -1).keys())
print(len(FOOTBALL_FEATURES), "features:", FOOTBALL_FEATURES)

In [ ]:
def build_football_examples(games, per_season=None):
    """One row per fixture. Slow by design — it mirrors production rather than vectorising,
    so the leakage guard stays visible.

    per_season samples evenly ACROSS seasons. A flat head(n) would take only the earliest
    fixtures and leave the validation and test splits empty, since the split is temporal."""
    rows = []
    grouped = list(games.groupby(["FIXTURE_ID", "SEASON"]))
    if per_season:
        by_season = {}
        for key, group in grouped:
            by_season.setdefault(key[1], []).append((key, group))
        grouped = [kg for season in sorted(by_season)
                   for kg in by_season[season][:per_season]]
    for (fid, season), group in grouped:
        home = group[group.HOME_AWAY == "home"]
        away = group[group.HOME_AWAY == "away"]
        if home.empty or away.empty:
            continue
        h, a = home.iloc[0], away.iloc[0]
        feats = football_features(games, h.TEAM_ID, a.TEAM_ID, h.GAME_DATE, fid)
        result = "home" if h.GF > h.GA else ("draw" if h.GF == h.GA else "away")
        rows.append({**feats, "SEASON": season, "FIXTURE_ID": fid,
                     "home_goals": h.GF, "away_goals": h.GA, "result": result})
    return pd.DataFrame(rows)

# Full build takes a few minutes; drop per_season for the real thing.
fb = build_football_examples(games, per_season=400)
print(fb.shape)
fb.result.value_counts(normalize=True).round(3)

In [ ]:
TRAIN, VAL, TEST = [2021, 2022, 2023], 2024, 2025
tr, va, te = temporal_split(fb, TRAIN, VAL, TEST)
print(f"train={len(tr)} val={len(va)} test={len(te)}")

Xtr, Xva, Xte = (d[FOOTBALL_FEATURES].astype(float) for d in (tr, va, te))

# --- Layer 1: two Poisson regressors -> expected goals per side ---
layer1_home = xgb.XGBRegressor(objective="count:poisson", n_estimators=200, max_depth=4)
layer1_away = xgb.XGBRegressor(objective="count:poisson", n_estimators=200, max_depth=4)
layer1_home.fit(Xtr, tr.home_goals.astype(float))
layer1_away.fit(Xtr, tr.away_goals.astype(float))

for name, d, X in (("train", tr, Xtr), ("test", te, Xte)):
    mae_h = np.abs(layer1_home.predict(X) - d.home_goals).mean()
    mae_a = np.abs(layer1_away.predict(X) - d.away_goals).mean()
    print(f"{name}: xg MAE home={mae_h:.4f} away={mae_a:.4f}")

In [ ]:
# --- Layer 2: 1X2 from Layer 1's xG plus context ---
CLASSES = ("home", "draw", "away")
LABEL = {c: i for i, c in enumerate(CLASSES)}
LAYER2_CONTEXT = ["form_pts_home", "form_pts_away", "h2h_win_rate_home",
                  "h2h_avg_goals_scored_home", "h2h_avg_goals_allowed_home",
                  "elo_diff", "win_streak_home", "win_streak_away"]

def layer2_matrix(X, d):
    out = pd.DataFrame(index=X.index)
    out["xg_home"] = layer1_home.predict(X)
    out["xg_away"] = layer1_away.predict(X)
    for c in LAYER2_CONTEXT:
        out[c] = d[c].astype(float).values
    return out

L2tr, L2va, L2te = layer2_matrix(Xtr, tr), layer2_matrix(Xva, va), layer2_matrix(Xte, te)
ytr, yva, yte = (d.result.map(LABEL).values for d in (tr, va, te))

layer2 = xgb.XGBClassifier(objective="multi:softprob", num_class=3,
                           n_estimators=200, max_depth=4)
layer2.fit(L2tr, ytr)

test_proba = layer2.predict_proba(L2te)
acc = accuracy_score(yte, test_proba.argmax(axis=1))
baseline = (te.result == "home").mean()
print(f"1X2 accuracy={acc:.4f}  always-home baseline={baseline:.4f}  edge={acc-baseline:+.4f}")

### Over/Under is **not** a trained model

It is the Poisson CDF of the two expected-goal rates. The sum of two independent Poissons is
itself Poisson, so `P(total ≤ line)` follows directly from `xg_home + xg_away`.

This is exactly why Over/Under inherits every weakness of the xG prediction — and why the
production model's Over/Under discrimination sits at **p=0.062**, short of significance.

In [ ]:
from math import exp, factorial

def under_probability(xg_home, xg_away, line):
    """P(total goals < line) for a half-integer line, via the summed Poisson."""
    lam = xg_home + xg_away
    return sum(exp(-lam) * lam**k / factorial(k) for k in range(int(np.floor(line)) + 1))

xg_h, xg_a = layer1_home.predict(Xte), layer1_away.predict(Xte)
totals = (te.home_goals + te.away_goals).values

print(f"{'line':>6}{'mean pred':>12}{'actual':>10}{'gap':>9}")
for line in (1.5, 2.5, 3.5, 4.5):
    p = np.array([under_probability(h, a, line) for h, a in zip(xg_h, xg_a)])
    actual = (totals < line).mean()
    print(f"{line:>6}{p.mean():>12.3f}{actual:>10.3f}{p.mean()-actual:>+9.3f}")

### Reliability buckets — the test that actually matters

Calibration (mean predicted ≈ actual) is easy and the model already passes it. What matters is
**discrimination**: do fixtures the model rates high genuinely go under more often?

If every bucket lands on the base rate, the model is calibrated but says nothing.

In [ ]:
line = 3.5
p = np.array([under_probability(h, a, line) for h, a in zip(xg_h, xg_a)])
actual = (totals < line).astype(int)
base = actual.mean()

print(f"base rate = {base:.3f}\n")
print(f"{'bucket':>12}{'n':>7}{'actual':>10}")
for lo in np.arange(0.4, 1.0, 0.1):
    mask = (p >= lo) & (p < lo + 0.1)
    if mask.sum() >= 20:
        print(f"{lo:.1f}-{lo+0.1:.1f}{mask.sum():>10}{actual[mask].mean():>10.3f}")

# Cochran-Armitage trend test: is the ordering real, or noise?
mids = np.arange(0.45, 1.0, 0.1)
ns = np.array([((p >= m - 0.05) & (p < m + 0.05)).sum() for m in mids])
rs = np.array([actual[(p >= m - 0.05) & (p < m + 0.05)].sum() for m in mids])
keep = ns > 0
mids, ns, rs = mids[keep], ns[keep], rs[keep]
xbar = (ns * mids).sum() / ns.sum()
z = (rs * (mids - xbar)).sum() / np.sqrt(base * (1 - base) * (ns * (mids - xbar) ** 2).sum())
print(f"\ntrend z = {z:+.2f}   (production run: z=+1.86, p=0.062 — short of significance)")

---
## 4. Tennis (ATP)

Same shape as NBA — one XGBoost binary classifier — because tennis is also two-outcome.

**The leakage bug worth knowing about.** BallDontLie always lists the eventual **winner** as
`player1` in a completed match (confirmed 20/20 in a sampled batch). Treating that as a neutral
positional label bakes the outcome into the label: the first real training run produced 100%
"home won" and XGBoost could not fit at all. The fix is an outcome-independent tiebreak —
**lower player id becomes home** — applied identically to scheduled and completed matches.

In [ ]:
tennis = pd.read_parquet(DATA / "tennis_game_log_atp.parquet")
ranks = pd.read_parquet(DATA / "tennis_rank_points_atp.parquet")
tennis["GAME_DATE"] = pd.to_datetime(tennis["GAME_DATE"])
# The log stores WL as "W"/"L"; every feature below wants a 0/1 column.
tennis["WON"] = (tennis["WL"] == "W").astype(int)
print(tennis.shape, ranks.shape)
print("columns:", list(tennis.columns))
tennis.head(3)

In [ ]:
def tennis_features(matches, home_id, away_id, as_of, surface=None):
    h = matches[matches.PLAYER_ID == home_id]
    a = matches[matches.PLAYER_ID == away_id]

    def days_since(pm):
        prior = pm[pm.GAME_DATE < as_of]
        return None if prior.empty else float((as_of - prior.GAME_DATE.max()).days)

    def streak(pm, surf=None):
        prior = pm[pm.GAME_DATE < as_of]
        if surf is not None and "SURFACE" in prior.columns:
            prior = prior[prior.SURFACE == surf]
        prior = prior.sort_values("GAME_DATE", ascending=False)
        run = 0
        for won in prior.WON:
            if not won:
                break
            run += 1
        return float(run)

    def surface_rate(pm, surf):
        if surf is None or "SURFACE" not in pm.columns:
            return None
        prior = pm[(pm.GAME_DATE < as_of) & (pm.SURFACE == surf)]
        return float(prior.WON.mean()) if len(prior) else None

    h2h = h[(h.OPPONENT_ID == away_id) & (h.GAME_DATE < as_of)]
    return {
        "form_win_rate_home": rolling_before(h, as_of, "WON", 10),
        "form_win_rate_away": rolling_before(a, as_of, "WON", 10),
        "days_since_last_match_home": days_since(h),
        "days_since_last_match_away": days_since(a),
        "win_streak_home": streak(h),
        "win_streak_away": streak(a),
        "h2h_win_rate_home": float(h2h.WON.mean()) if len(h2h) else None,
        "surface_win_rate_home": surface_rate(h, surface),
        "surface_win_rate_away": surface_rate(a, surface),
        "surface_streak_home": streak(h, surface),
        "surface_streak_away": streak(a, surface),
    }

print("tennis feature keys:", list(tennis_features(tennis, tennis.PLAYER_ID.iloc[0],
      tennis.OPPONENT_ID.iloc[0], tennis.GAME_DATE.max()).keys()))

> **`rank_diff`** comes from `tennis_rank_points_atp.parquet` (PLAYER_ID, WEEK, RANK_POINTS),
> joined on the nearest ranking week **before** the match — same leakage rule as everything
> else. It is the single strongest tennis feature, standing in for the Elo that football uses.

---
## 5. Key-player availability — and the leakage trap inside it

`team_key_players.parquet` is **Stage 1**: each team's top players per season, ranked by a
quality metric. Turning that into a feature has two completely different definitions, and they
must never share code:

| | Question | Knowable | Correct use |
|---|---|---|---|
| **Training label** | Did they actually play? | Only *after* the match | Backtesting |
| **Live serving** | Are they injured? | *Before* the match | Forecasting |

Using the training definition to forecast is target leakage. The production codebase keeps them
in separate modules specifically so an import cannot blur the line, and a regression test
asserts they disagree on the same underlying facts.

The notebook below uses the **training-label** form, which is correct here and wrong live.

In [ ]:
key_players = pd.read_parquet(DATA / "team_key_players.parquet")
print(key_players.shape)
print(key_players.head(5).to_string(index=False))

def key_players_available(lineups, key_players, fixture_id, team_id, season):
    """TRAINING-LABEL form: how many of this team's top players actually appeared.

    Correct for a backtest, invalid as a forecast — the answer does not exist before kick-off.
    """
    top = set(key_players[(key_players.TEAM_ID.astype(str) == str(team_id))
                          & (key_players.SEASON == season)].PLAYER_NAME.str.lower())
    if not top:
        return None, None
    played = set(lineups[(lineups.FIXTURE_ID == fixture_id)
                         & (lineups.TEAM_ID.astype(str) == str(team_id))]
                 .PLAYER_NAME.str.lower())
    available = top & played
    metric = key_players[(key_players.TEAM_ID.astype(str) == str(team_id))
                         & (key_players.SEASON == season)
                         & (key_players.PLAYER_NAME.str.lower().isin(available))]
    return float(len(available)), float(metric.COMBINED_METRIC.sum())

---
## 6. What the production runs found

Reproduced here so the notebook's numbers can be compared against them.

| Model | Test accuracy | Baseline | Edge |
|---|---|---|---|
| NBA | **68.57%** | 55.43% (always home) | **+13.1 pts** |
| Tennis | 63.86% | 62.22% (higher-ranked) | +1.6 pts |
| Football 1X2 | 47.75% | 45.60% (always home) | +2.2 pts |

### Over/Under: three attempts, honestly recorded

| Attempt | Outcome |
|---|---|
| Negative Binomial for overdispersion | **Refuted before building.** Measured var/mean = 1.0030 across 8,718 fixtures — the dispersion does not exist. corr(home, away goals) = −0.058, so Dixon-Coles is equally unjustified. |
| Rolling xG | **Modest gain.** Trend z: +0.73 → **+1.86**. Buckets went flat → monotonic. Still p=0.062. |
| League-identity features | **Regression.** z collapsed to −0.03, 1X2 fell 0.4775 → 0.4634, buckets inverted. Built, measured, disabled. |

### Why Over/Under resists

Feature importance on Layer 1 tells the story: `elo_diff` dominates at **12.5%**, ~2.6× the
next feature — and Elo measures who is *stronger*, not how many goals get scored. Everything
else sits at 0.030–0.048, which is what a model with no decisive signal looks like.

The feature set is largely **outcome-predictive** signals being asked for a **volume**
prediction.

### Things to try in this notebook

1. **Shot-based features** — shots, shots on target, big chances are volume measures. The last
   untried lever, and the data is already collected.
2. **Out-of-fold Layer 2** — Layer 2 currently trains on Layer 1's in-sample predictions, which
   makes its training signal slightly optimistic.
3. **Historical odds** — the market price is the strongest single predictor in sports betting
   and is currently populated on 0.7% of rows.
4. **Do not trust the ROI numbers.** No sample yet exceeds n=49.